In [ ]:
pip install selenium beautifulsoup4 pandas tqdm
pip install webdriver_manager

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

# 0. 셀레니움 드라이버 설정
options = Options()
# options.add_argument("--headless")  # 필요시 주석 해제
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--no-sandbox')
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# 1. 법정동 리스트 로드
df_dong = pd.read_csv("법정동_리스트.csv")
dong_list = df_dong[["법정동코드", "법정동"]].drop_duplicates().values.tolist()

# 2. 크롤링 결과와 실패 리스트 초기화
all_results = []
failed_dongs = []

# 3. 전체 동 반복 크롤링
for code, dong_name in dong_list:
    url = f"https://hogangnono.com/region/{code}/0/apt-list"
    try:
        driver.get(url)
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        section = soup.select_one("#local3-aptlist-scroll")
        if not section:
            print(f"[❌ 없음] {dong_name} (코드: {code}) - 리스트 영역 없음")
            failed_dongs.append((code, dong_name))
            continue

        category_blocks = section.select("div.css-jsrvbw.ekplin50")
        apt_list = []

        for block in category_blocks:
            category = block.select_one("h3.type").text.strip()
            links = block.select("ul.region-list li a")
            for link in links:
                name_tag = link.select_one("h5")
                href = link.get("href")
                if name_tag and href:
                    apt_list.append({
                        "name": name_tag.text.strip(),
                        "url": f"https://hogangnono.com{href}",
                        "category": category
                    })

        # 각 단지별 상세정보 크롤링
        for apt in apt_list:
            driver.get(apt["url"])
            time.sleep(2)

            try:
                addr = wait.until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.text-sm.font-semibold.text-foreground"))
                ).text.strip()
            except:
                addr = None

            # 요약 버튼 클릭
            try:
                top_button = wait.until(
                    EC.element_to_be_clickable((
                        By.CSS_SELECTOR,
                        "#page-header > div > div.absolute.bottom-0.left-0.right-0.h-12.bg-primary.px-5.z-layer.text-background > div > div > div > div.flex.flex-1 > button"
                    ))
                )
                driver.execute_script("arguments[0].click();", top_button)
                time.sleep(2)

                # 요약글
                summaries = driver.find_elements(By.CSS_SELECTOR, "p.px-5.text-base.text-foreground")
                summary = summaries[0].text.strip() if summaries else "요약글 없음"

                # 해시태그
                tag_elements = driver.find_elements(By.CSS_SELECTOR,
                    "#reviewPage-scroll > div > section:nth-child(3) > div.mt-4.flex.flex-wrap.gap-2.px-5 > a")
                tags = [tag.text.strip() for tag in tag_elements] if tag_elements else []

            except Exception as e:
                summary = "요약글 없음"
                tags = []
                print(f"[요약/태그 실패] {apt['name']} ({dong_name}) : {e}")

            all_results.append({
                "법정동": dong_name,
                "단지명": apt["name"],
                "카테고리": apt["category"],
                "주소": addr,
                "요약글": summary,
                "해시태그": ", ".join(tags)
            })

        print(f"✅ 완료: {dong_name} ({code})")

    except Exception as e:
        print(f"[❌ 실패] {dong_name} (코드: {code}) - {e}")
        failed_dongs.append((code, dong_name))
        continue

# 4. 저장
driver.quit()
df_all = pd.DataFrame(all_results)
df_all.to_csv("서울시_전체단지_요약_해시태그.csv", index=False, encoding="utf-8-sig")

# 5. 실패 동 출력
if failed_dongs:
    print("\n❌ 크롤링 실패한 동 목록:")
    for code, name in failed_dongs:
        print(f"- {name} ({code})")
else:
    print("\n✅ 모든 법정동 크롤링 성공!")